## Clases abstractas

In [2]:
class RungeKutta:
    def __init__(self, system, h):
        """Inicializa la clase RungeKutta."""
        self.system = system
        self.h = h

    def step(self, t, y, h=None):
        """Realiza un solo paso del método de Runge-Kutta de cuarto orden."""
        if h is None:
            h = self.h  # Usa h de la instancia si no se proporciona uno específico
        
        k1 = self.system(t, y)
        k2 = self.system(t + h/2, [y[i] + h/2 * k1[i] for i in range(len(y))])
        k3 = self.system(t + h/2, [y[i] + h/2 * k2[i] for i in range(len(y))])
        k4 = self.system(t + h, [y[i] + h * k3[i] for i in range(len(y))])

        return [y[i] + (h/6) * (k1[i] + 2*k2[i] + 2*k3[i] + k4[i]) for i in range(len(y))]

    def solve(self, t_span, y0):
        """Resuelve el sistema de ecuaciones diferenciales usando Runge-Kutta."""
        t_start, t_end = t_span
        num_steps = int((t_end - t_start) / self.h) + 1  # Cantidad de pasos
        t_values = [t_start + i * self.h for i in range(num_steps)]  # Lista de tiempos predefinida
        y_values = [y0]

        y = y0[:]  # Copia de las condiciones iniciales

        for i in range(1, num_steps):
            if t_values[i] > t_end:  # Ajustar último paso si es necesario
                h_last = t_end - t_values[i-1]
                y = self.step(t_values[i-1], y, h_last)
                t_values[i] = t_end
            else:
                y = self.step(t_values[i-1], y)
            
            y_values.append(y[:])  # Evitar mutabilidad

        return t_values, y_values

In [4]:
import math
import matplotlib.pyplot as plt
import RungeKutta  # Asegúrate de tener tu implementación de Runge-Kutta

# Parámetros físicos
mu = 1  # Constante gravitacional
L = 1   # Momento angular específico
K = 2   # Constante de fuerza inversa al cuadrado
h = 0.01  # Paso de integración
psi_start = 0  # Ángulo inicial
psi_end = 10   # Ángulo final
z0 = 1   # Condición inicial: z(0) = 1/r(0)
w0 = -0.1  # Condición inicial: dz/dψ(0)

# Definir el sistema de ecuaciones diferenciales
def system(psi, y):
    z, w = y
    dz_dpsi = w
    dw_dpsi = -z + (mu * K) / (L**2)
    return [dz_dpsi, dw_dpsi]

# Crear una instancia de RungeKutta y resolver
rk = RungeKutta(system, h)
t_span = (psi_start, psi_end)
y0 = [z0, w0]
psi_values, y_values = rk.solve(t_span, y0)

# Extraer soluciones
z_values = [y[0] for y in y_values]

# Convertir de z a r (r = 1/z)
r_values = [1 / z if z != 0 else float('inf') for z in z_values]

# Convertir ángulos psi a coordenadas cartesianas sin NumPy
x_values = [r * math.cos(psi) for r, psi in zip(r_values, psi_values)]
y_values = [r * math.sin(psi) for r, psi in zip(r_values, psi_values)]

# Graficar la órbita en coordenadas cartesianas
plt.figure(figsize=(8, 8))
plt.plot(x_values, y_values, label="Órbita del cuerpo", color="blue")
plt.scatter(0, 0, color="red", marker="o", label="Cuerpo central")  # Representa el foco
plt.xlabel("x")
plt.ylabel("y")
plt.title("Órbita del cuerpo en coordenadas cartesianas")
plt.legend()
plt.grid()
plt.axis("equal")  # Mantiene la escala 1:1 en los ejes
plt.show()

ModuleNotFoundError: No module named 'RungeKutta'

In [1]:
from abc import ABC, abstractmethod
class CuerpoCeleste(ABC):
    def __init__(self, masa, x, y, vx, vy):
        self._masa = masa
        self._x = x
        self._y = y
        self._vx = vx
        self._vy = vy
        self.trayectoria = []  # Lista para almacenar la trayectoria

    @property
    def masa(self):
        return self._masa

    @property
    def posicion(self):
        return self._x, self._y

    @property
    def velocidad(self):
        return self._vx, self._vy

    @abstractmethod
    def actualizar_posicion(self):
        pass

    def agregar_trayectoria(self):
        # Almacenar la posición actual de la partícula
        self.trayectoria.append((self._x, self._y))


In [8]:
from functools import lru_cache
class Planeta(CuerpoCeleste):
    def __init__(self, masa, x, y, vx, vy, epsilon):
        super().__init__(masa, x, y, vx, vy)
        self.epsilon = epsilon  # Excentricidad

    @lru_cache(None)
    def calcular_orbita(self, alpha, psi):
        """
        Calcula la órbita según la ley de Kepler y la excentricidad epsilon.
        """
        denominador = 1 + self.epsilon * math.cos(psi)
        if abs(denominador) < 1e-6:
            return None
        return alpha / denominador

    def actualizar_posicion(self, alpha, psi):
        r = self.calcular_orbita(alpha, psi)
        if r is not None:
            self._x = r * math.cos(psi)
            self._y = r * math.sin(psi)
        self.agregar_trayectoria()  # Agregar la nueva posición a la trayectoria
       # Decorador para medir el tiempo de ejecución 
def timing(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"Tiempo de ejecución de {func.__name__}: {end_time - start_time:.4f} segundos")
        return result
    return wrapper


In [6]:
#función de animación
import time
@timing
def animar_orbitas(planetas, alpha, psi_vals, velocidad_factor=1):
    fig, ax = plt.subplots(figsize=(7, 7))

    def update(num):
        ax.clear()
        # Aquí aumentamos el valor de psi para acelerar el avance de las partículas
        velocidad_psi = psi_vals[num] * velocidad_factor  # Esto hace que las partículas se muevan más rápido
        for planeta in planetas:
            planeta.actualizar_posicion(alpha, velocidad_psi)

            # Graficar la trayectoria de la partícula (todas las posiciones anteriores)
            trayectoria_x, trayectoria_y = zip(*planeta.trayectoria)  # Extraer las posiciones
            ax.plot(trayectoria_x, trayectoria_y, linestyle='-', color='gray', alpha=0.6)  # Trayectoria

            # Graficar la posición actual de la partícula
            ax.plot(planeta.posicion[0], planeta.posicion[1], 'bo')

        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.axhline(0, color='black', linewidth=0.5)
        ax.axvline(0, color='black', linewidth=0.5)
        ax.grid(True, linestyle="--", linewidth=0.5)

    ani = animation.FuncAnimation(fig, update, frames=len(psi_vals), interval=30)  # Menor intervalo
    plt.show()
